# PPE 탐지 성능 테스트 — yolo26 (코랩 학습)

상용화 전 성능 테스트용. 공개 데이터셋(SH17 + Construction Site Safety)을 **통합 8클래스**로
리매핑해 yolo26 을 학습하고, 현행 라이브 모델(`vyra_ppe.pt`)과 비교한다.

**통합 클래스** — `person · head · helmet · safety_vest · foot · shoes · hands · gloves`

미착용은 별도 클래스로 두지 않고 `head`(맨머리) + 포함관계로 코드에서 판정한다.
근거는 [context-notes.md](context-notes.md).

## 이 노트북이 하지 않는 것

- **TensorRT 엔진 변환** — 코랩은 x86 + 데이터센터 GPU라 여기서 만든 `.engine` 은 Jetson 에서 못 쓴다.
  엔진 변환·INT8 캘리브레이션은 **타깃 Jetson 에서** 한다. 코랩 산출물은 `.pt` 까지다.
- **이벤트 단위 오탐 측정** — 실 현장 영상이 있어야 한다.

## ⚠️ 배포 타깃 제약 (Jetson Orin Nano 4GB · 실사용 3.5GB)

메모리가 지배 제약이다. 학습(코랩)은 자유롭지만 **배포 후보는 `yolo26s` 부터** 보는 것이 현실적이다.
`yolo26m` 은 정확도 상한 측정용으로 함께 학습해 두고, 젯슨 실측으로 판정한다.


## 1. 런타임 확인

런타임 유형이 GPU 인지 먼저 확인한다. `런타임 → 런타임 유형 변경 → T4 GPU`.

In [ ]:
!nvidia-smi
!free -g | head -2
import sys; print('python', sys.version.split()[0])

In [ ]:
%pip -q install "ultralytics>=8.4.0" kaggle roboflow
import ultralytics; ultralytics.checks()

## 2. 설정

세션이 끊겨도 체크포인트가 남도록 Drive 에 저장한다. 재개는 `RESUME = True`.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# ---- 설정 -------------------------------------------------------------
MODEL      = 'yolo26s.pt'   # 배포 후보. 정확도 상한 측정은 'yolo26m.pt' 로 한 번 더 돌린다
IMGSZ      = 960            # 안전화·장갑 같은 소형 객체 때문. 640 과 비교 학습할 것
EPOCHS     = 150
BATCH      = -1             # -1 = AutoBatch(VRAM 자동). OOM 나면 8 로 고정
USE_SH17   = True           # ⚠️ CC BY-NC-SA(비상업) — 성능 테스트 한정, 상용 빌드 반입 금지
USE_CSS    = True           # CC BY 4.0 — 상용 반입 가능
RESUME     = False          # 세션 끊긴 뒤 이어서 학습할 때 True
# -----------------------------------------------------------------------

RUN_NAME  = f"{Path(MODEL).stem}_{IMGSZ}" + ("_nc" if USE_SH17 else "")
DRIVE     = Path('/content/drive/MyDrive/vitalguard-ppe')
RUNS      = DRIVE / 'runs'
RAW       = Path('/content/raw')       # 원본 데이터셋(세션 로컬 — 재다운로드 가능)
DATA      = Path('/content/ppe')       # 통합 데이터셋
DATA_YAML = DATA / 'data.yaml'
for p in (DRIVE, RUNS, RAW, DATA):
    p.mkdir(parents=True, exist_ok=True)

print('run   :', RUN_NAME)
print('runs  :', RUNS)
if USE_SH17:
    print('\n⚠️  SH17(CC BY-NC-SA) 포함 — 산출 가중치는 비상업. 파일명 -nc 접미사 유지할 것.')

## 3. 데이터셋 다운로드

- **SH17** — Kaggle. `kaggle.json`(계정 → Settings → API → Create New Token) 업로드 필요.
- **CSS(Construction Site Safety)** — Roboflow. API 키 필요.

둘 다 계정이 필요하다. 없으면 해당 플래그를 `False` 로 두고 한쪽만 써도 파이프라인은 돈다.

In [ ]:
# --- SH17 (Kaggle) ---
if USE_SH17:
    import os
    if not Path('/root/.config/kaggle/kaggle.json').exists():
        from google.colab import files
        print('kaggle.json 을 업로드하세요.')
        files.upload()
        os.makedirs('/root/.config/kaggle', exist_ok=True)
        !cp kaggle.json /root/.config/kaggle/ && chmod 600 /root/.config/kaggle/kaggle.json
    !kaggle datasets download -d mugheesahmad/sh17-dataset-for-ppe-detection -p /content/raw --unzip -q
    print('SH17 다운로드 완료')

In [ ]:
# --- CSS (Roboflow) ---
if USE_CSS:
    from getpass import getpass
    from roboflow import Roboflow
    rf = Roboflow(api_key=getpass('Roboflow API key: '))
    ds = (rf.workspace('roboflow-universe-projects')
            .project('construction-site-safety')
            .version(27)
            .download('yolov8', location='/content/raw/css'))
    print('CSS 다운로드 완료 →', ds.location)

In [ ]:
# ★ 실제 디렉터리 구조와 클래스 순서를 눈으로 확인한다.
#   데이터셋 배포본은 버전마다 레이아웃이 다르므로 여기서 반드시 대조할 것.
!find /content/raw -maxdepth 3 -type d | head -40
print()
!find /content/raw -name 'data.yaml' -o -name '*.yaml' | head
print()
for y in Path('/content/raw').rglob('*.yaml'):
    print('---', y)
    print(y.read_text()[:600])

## 4. 통합 8클래스로 리매핑

클래스 인덱스가 아니라 **이름으로 매핑**한다. 배포본마다 인덱스 순서가 달라질 수 있기 때문이다.
원본 `data.yaml` 의 `names` 를 읽어 인덱스를 그때그때 해석한다.

In [ ]:
import hashlib, shutil, yaml
from collections import Counter
from pathlib import Path

UNIFIED = ['person', 'head', 'helmet', 'safety_vest', 'foot', 'shoes', 'hands', 'gloves']
UNIFIED_IDX = {n: i for i, n in enumerate(UNIFIED)}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def _norm(s):
    """클래스명 비교용 정규화 — 대소문자·공백·하이픈·언더바 무시."""
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())


# 원본 클래스명(정규화) → 통합 클래스명. 표에 없는 클래스는 버린다.
SH17_MAP = {
    'person': 'person', 'head': 'head', 'helmet': 'helmet',
    'safetyvest': 'safety_vest', 'foot': 'foot', 'shoes': 'shoes',
    'hands': 'hands', 'gloves': 'gloves',
}
# NO-Hardhat(헬멧 없는 머리) 은 head 로 흡수 — 미착용을 별도 클래스로 두지 않는 설계.
CSS_MAP = {
    'person': 'person', 'hardhat': 'helmet', 'nohardhat': 'head',
    'safetyvest': 'safety_vest',
}


def load_names(data_yaml):
    d = yaml.safe_load(Path(data_yaml).read_text())
    names = d['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    return list(names)


def build_index_map(src_names, name_map):
    """원본 클래스 인덱스 → 통합 인덱스. 매핑 없는 클래스는 None."""
    out = {}
    for i, n in enumerate(src_names):
        tgt = name_map.get(_norm(n))
        out[i] = UNIFIED_IDX[tgt] if tgt else None
    return out


def deterministic_split(stem, val_ratio=0.1):
    """파일명 해시로 train/val 을 고정 배정 — 재실행해도 동일 분할."""
    h = int(hashlib.md5(stem.encode()).hexdigest()[:8], 16)
    return 'val' if (h % 1000) < val_ratio * 1000 else 'train'


def collect_pairs(root, val_ratio=0.1):
    """root 아래 images/ 를 훑어 (이미지, 라벨, split) 목록을 만든다.

    경로에 train/val/valid/test 가 있으면 그것을 쓰고, 없으면 파일명 해시로 분할한다.
    """
    root = Path(root)
    pairs = []
    for img in sorted(root.rglob('*')):
        if img.suffix.lower() not in IMG_EXT or not img.is_file():
            continue
        parts_lower = [p.lower() for p in img.parts]
        if 'images' not in parts_lower:
            continue
        i = len(parts_lower) - 1 - parts_lower[::-1].index('images')
        lbl = Path(*img.parts[:i], 'labels', *img.parts[i + 1:]).with_suffix('.txt')
        s = set(parts_lower)
        if 'train' in s:
            sp = 'train'
        elif s & {'val', 'valid'}:
            sp = 'val'
        elif 'test' in s:
            sp = 'test'
        else:
            sp = deterministic_split(img.stem, val_ratio)
        pairs.append((img, lbl, sp))
    return pairs


def remap(pairs, dst_root, idx_map, prefix, link=True):
    """라벨을 통합 인덱스로 다시 쓰고 이미지를 dst_root 로 모은다.

    남는 박스가 하나도 없는 이미지는 건너뛴다(순수 배경 이미지 범람 방지).
    반환 = (유지 수, 제외 수, 클래스별 인스턴스 수)
    """
    dst_root = Path(dst_root)
    kept = dropped = 0
    counts = Counter()
    for img, lbl, sp in pairs:
        lines = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                try:
                    new = idx_map.get(int(float(parts[0])))
                except ValueError:
                    continue
                if new is None:
                    continue
                lines.append(' '.join([str(new)] + parts[1:5]))
                counts[UNIFIED[new]] += 1
        if not lines:
            dropped += 1
            continue
        img_out = dst_root / 'images' / sp
        lbl_out = dst_root / 'labels' / sp
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        stem = f'{prefix}_{img.stem}'
        dst_img = img_out / (stem + img.suffix)
        if not dst_img.exists():
            if link:
                try:
                    dst_img.symlink_to(img.resolve())
                except OSError:
                    shutil.copy2(img, dst_img)
            else:
                shutil.copy2(img, dst_img)
        (lbl_out / (stem + '.txt')).write_text('\n'.join(lines) + '\n')
        kept += 1
    return kept, dropped, counts


print('리매핑 유틸 로드 완료')

In [ ]:
# 원본 경로와 data.yaml 은 위 4번 셀 출력으로 확인한 실제 경로로 맞출 것.
SOURCES = []
if USE_SH17:
    SOURCES.append(dict(name='sh17', root='/content/raw',
                        yaml=None, names=None, mapping=SH17_MAP))
if USE_CSS:
    SOURCES.append(dict(name='css', root='/content/raw/css',
                        yaml='/content/raw/css/data.yaml', names=None, mapping=CSS_MAP))

# SH17 은 data.yaml 이 없을 수 있어 공식 클래스 순서를 명시(위 출력과 대조해 필요 시 수정)
SH17_NAMES = ['person', 'head', 'face', 'glasses', 'face-mask-medical', 'face-guard',
              'ear', 'earmuffs', 'hands', 'gloves', 'foot', 'shoes', 'safety-vest',
              'tools', 'helmet', 'medical-suit', 'safety-suit']

total = Counter()
for s in SOURCES:
    names = load_names(s['yaml']) if s['yaml'] else (s['names'] or SH17_NAMES)
    idx_map = build_index_map(names, s['mapping'])
    mapped = {names[i]: UNIFIED[v] for i, v in idx_map.items() if v is not None}
    print(f"\n[{s['name']}] 원본 {len(names)}클래스 → 유지 {len(mapped)}종")
    print('  ', mapped)
    pairs = collect_pairs(s['root'])
    print(f'   이미지 {len(pairs)}장 수집')
    kept, dropped, counts = remap(pairs, DATA, idx_map, s['name'])
    print(f'   유지 {kept} / 제외 {dropped}(매핑 대상 박스 없음)')
    print('  ', dict(counts))
    total.update(counts)

print('\n=== 통합 데이터셋 클래스별 인스턴스 ===')
for n in UNIFIED:
    c = total.get(n, 0)
    flag = '  ⚠️ 부족' if 0 < c < 500 else ('  ⚠️ 없음' if c == 0 else '')
    print(f'  {n:<12} {c:>7,}{flag}')

In [ ]:
import yaml

splits = {p.name: len(list(p.glob('*'))) for p in (DATA / 'images').iterdir() if p.is_dir()}
print('split 별 이미지 수:', splits)

cfg = {'path': str(DATA),
       'train': 'images/train',
       'val': 'images/val',
       'names': {i: n for i, n in enumerate(UNIFIED)}}
if 'test' in splits:
    cfg['test'] = 'images/test'
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print()
print(DATA_YAML.read_text())

## 5. 학습

체크포인트는 Drive 에 저장된다. 세션이 끊기면 위 설정 셀에서 `RESUME = True` 로 바꾸고 이 셀만 다시 실행한다.

In [ ]:
from ultralytics import YOLO

last = RUNS / RUN_NAME / 'weights' / 'last.pt'
if RESUME and last.exists():
    print('재개:', last)
    model = YOLO(str(last))
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        project=str(RUNS), name=RUN_NAME, exist_ok=True,
        patience=30, workers=2, seed=0, plots=True,
    )

best = RUNS / RUN_NAME / 'weights' / 'best.pt'
print('\nbest:', best)

## 6. 평가 — 1층 검출 지표

전체 mAP 는 큰 객체(person)에 묻힌다. **클래스별로** 봐야 안전화·장갑의 실태가 드러난다.

In [ ]:
import pandas as pd
from ultralytics import YOLO


def per_class(weights, data_yaml=None, imgsz=None, split='val'):
    m = YOLO(str(weights))
    r = m.val(data=str(data_yaml or DATA_YAML), imgsz=imgsz or IMGSZ, split=split, verbose=False)
    names = r.names
    rows = []
    for i, c in enumerate(r.box.ap_class_index):
        rows.append({'class': names[int(c)],
                     'P': round(float(r.box.p[i]), 3),
                     'R': round(float(r.box.r[i]), 3),
                     'mAP50': round(float(r.box.ap50[i]), 3),
                     'mAP50-95': round(float(r.box.ap[i]), 3)})
    df = pd.DataFrame(rows).sort_values('R')
    return df, r


df, r = per_class(best)
print(f'전체 mAP50-95 = {float(r.box.map):.3f}   mAP50 = {float(r.box.map50):.3f}\n')
display(df)
print('\n※ recall 이 낮은 클래스가 위. 소형 객체(shoes·gloves·foot)가 하위면 imgsz 를 올려 재학습.')

## 7. 평가 — 2층 착용 판정

검출 mAP 가 좋아도 person ↔ helmet 매칭에서 깨질 수 있다. 착용 여부는 별도로 측정한다.

판정 규칙
- 장비 박스가 작업자 박스에 `CONTAIN_MIN` 이상 들어와야 그 사람 것으로 귀속한다.
- 안전모/맨머리는 작업자 박스 **상단 `HEAD_ZONE` 안에 중심**이 있어야 한다(허리춤에 든 헬멧 배제).
- 안전모 위반은 **맨머리(`head`)가 보일 때만** 센다. 뒤통수만 보여 아무것도 안 잡힌 경우를
  위반으로 올리면 오탐이 폭증한다.

In [ ]:
CONTAIN_MIN = 0.5   # 장비 박스가 작업자 박스에 이만큼 이상 들어와야 그 사람 것으로 본다
HEAD_ZONE   = 0.4   # 안전모는 작업자 박스 상단 40% 안에 중심이 있어야 한다


def _contain_ratio(inner, outer):
    """inner 박스가 outer 박스에 포함된 면적 비율. 둘 다 (x1,y1,x2,y2)."""
    ix1, iy1 = max(inner[0], outer[0]), max(inner[1], outer[1])
    ix2, iy2 = min(inner[2], outer[2]), min(inner[3], outer[3])
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    area = max(1e-9, (inner[2] - inner[0]) * (inner[3] - inner[1]))
    return inter / area


def judge_ppe(dets, contain_min=CONTAIN_MIN, head_zone=HEAD_ZONE):
    """검출 결과를 작업자 단위 착용 여부로 묶는다.

    dets = [(cls_name, (x1,y1,x2,y2), conf), ...]
    반환 = [{'box':..., 'helmet':bool, 'vest':bool, 'bare_head':bool}, ...]
    """
    out = []
    for name0, pb, _ in [d for d in dets if d[0] == 'person']:
        head_limit = pb[1] + (pb[3] - pb[1]) * head_zone
        rec = {'box': pb, 'helmet': False, 'vest': False, 'bare_head': False}
        for name, b, _c in dets:
            if name not in ('helmet', 'head', 'safety_vest'):
                continue
            if _contain_ratio(b, pb) < contain_min:
                continue
            cy = (b[1] + b[3]) / 2
            if name in ('helmet', 'head') and cy > head_limit:
                continue
            if name == 'helmet':
                rec['helmet'] = True
            elif name == 'head':
                rec['bare_head'] = True
            else:
                rec['vest'] = True
        out.append(rec)
    return out


def violations(judged):
    """착용 판정 결과 → 위반 건수. 안전모는 맨머리가 보일 때만 위반으로 센다."""
    return {'persons': len(judged),
            'no_helmet': sum(1 for r in judged if r['bare_head'] and not r['helmet']),
            'no_vest': sum(1 for r in judged if not r['vest'])}


def dets_from_result(res):
    names = res.names
    return [(names[int(c)], tuple(map(float, xyxy)), float(cf))
            for xyxy, c, cf in zip(res.boxes.xyxy.tolist(),
                                   res.boxes.cls.tolist(),
                                   res.boxes.conf.tolist())]


# val 이미지 전체에 적용해 위반 분포를 본다(정답 대비 정확도는 라벨과 대조해 별도 산출).
m = YOLO(str(best))
val_imgs = sorted((DATA / 'images' / 'val').glob('*'))
agg = Counter()
for res in m.predict(val_imgs[:300], imgsz=IMGSZ, conf=0.25, stream=True, verbose=False):
    agg.update(violations(judge_ppe(dets_from_result(res))))
print('val 300장 집계:', dict(agg))

## 8. 기준선 비교 — 현행 라이브 모델

판정 기준은 "얼마나 좋은가"가 아니라 **"현행보다 나은가"** 다.
현행 CAM-03 은 Hexmon `vyra_ppe.pt`(YOLOv8m 14클래스)를 쓴다.

클래스 구성이 달라 mAP 를 그대로 비교할 수 없으므로, **같은 이미지에 두 모델을 태워
안전모/조끼 착용 판정 결과를 대조**하는 방식으로 비교한다.

In [ ]:
# 현행 모델 내려받기 (레포에는 미추적 — cctv-ai/README.md 재취득 절차와 동일 출처)
!curl -sL -o /content/vyra_ppe.pt https://huggingface.co/Hexmon/vyra-yolo-ppe-detection/resolve/main/best.pt
baseline = YOLO('/content/vyra_ppe.pt')
print('현행 모델 클래스:', baseline.names)

# 현행 모델의 클래스명 → 우리 통합 이름으로 정규화 (실제 이름을 위 출력으로 확인 후 조정)
BASE_ALIAS = {'hardhat': 'helmet', 'helmet': 'helmet', 'nohardhat': 'head',
              'safetyvest': 'safety_vest', 'vest': 'safety_vest',
              'nosafetyvest': None, 'person': 'person', 'worker': 'person'}


def dets_baseline(res):
    out = []
    for xyxy, c, cf in zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist(), res.boxes.conf.tolist()):
        tgt = BASE_ALIAS.get(_norm(res.names[int(c)]))
        if tgt:
            out.append((tgt, tuple(map(float, xyxy)), float(cf)))
    return out


agg_b = Counter()
for res in baseline.predict(val_imgs[:300], imgsz=640, conf=0.25, stream=True, verbose=False):
    agg_b.update(violations(judge_ppe(dets_baseline(res))))

print('\n같은 val 300장')
print('  신규 yolo26 :', dict(agg))
print('  현행 vyra   :', dict(agg_b))
print('\n※ 위반 건수만으로는 우열을 못 가린다. 정답 라벨과 대조한 판정 정확도로 결론낼 것.')

## 9. 내보내기 → Jetson 이관

**코랩에서 TensorRT 엔진을 만들지 않는다.** 엔진은 기기·TensorRT 버전 종속이라 Jetson 에서 못 쓴다.
`.pt` 만 내보내고 변환은 젯슨에서 한다.

In [ ]:
import shutil

out = DRIVE / 'weights'
out.mkdir(exist_ok=True)
dst = out / f'ppe_{RUN_NAME}.pt'      # RUN_NAME 에 SH17 사용 시 -nc 접미사가 포함됨
shutil.copy2(best, dst)
print('저장:', dst, f'({dst.stat().st_size/1e6:.1f} MB)')
if USE_SH17:
    print('\n⚠️ 이 가중치는 SH17(비상업) 학습분이다. 상용 빌드에 반입하지 말 것.')

print(f'''
다음 — Jetson Orin Nano 4GB 에서 (메모리가 지배 제약):

  # 0) 엔진 빌드는 추론보다 메모리를 훨씬 많이 쓴다. 스왑 먼저.
  sudo systemctl disable nvzramconfig || true
  sudo fallocate -l 16G /var/swapfile && sudo chmod 600 /var/swapfile
  sudo mkswap /var/swapfile && sudo swapon /var/swapfile
  # 데스크톱 GUI 끄기(~800MB 확보)
  sudo systemctl set-default multi-user.target && sudo reboot

  # 1) 다른 워크로드(ROS2·푸셔) 내려두고 변환
  sudo nvpmodel -m 0 && sudo jetson_clocks
  python3 -c "from ultralytics import YOLO; YOLO('ppe_{RUN_NAME}.pt').export(format='engine', half=True, imgsz={IMGSZ}, workspace=2)"

  # 2) 정상상태 측정 (초기 30초 값은 무의미 — 스로틀링 후로 볼 것)
  yolo val model=ppe_{RUN_NAME}.engine data=... imgsz={IMGSZ}
  jtop   # 메모리·전력·온도 동시 확인
''')